# Pelatihan Model Abjad Jari & Angka SIBI (Landmark MediaPipe)

Pipeline: **gambar / rekaman → 21 landmark tangan → normalisasi → augmentasi → MLP kecil → evaluasi → ekspor JSON** (dipakai di Python lokal & browser tanpa library tambahan).

**Struktur data yang diharapkan** (upload ke Colab atau mount Google Drive):

```
/content/data/
├── sibi_mendeley/          <- satu folder per SUMBER dataset
│   ├── A/ *.jpg
│   ├── B/ *.jpg
│   └── ...
├── sibi_kaggle/
│   └── A/ ...
└── rekaman_budi.csv        <- hasil record_and_test.py (opsional)
```

Catatan penting:
- Pakai folder **gambar asli**, bukan folder yang sudah diaugmentasi oleh pemilik dataset (mencegah kebocoran data antara train dan test).
- J dan Z dikecualikan secara default karena merupakan isyarat bergerak.
- Runtime CPU sudah cukup. Bagian paling lama adalah ekstraksi landmark, bukan pelatihan.

In [ ]:
!pip -q install mediapipe
!wget -q -O hand_landmarker.task https://storage.googleapis.com/mediapipe-models/hand_landmarker/hand_landmarker/float16/1/hand_landmarker.task
# Jika muncul error versi numpy/protobuf setelah install: Runtime > Restart session, lalu jalankan ulang dari sel berikutnya.

In [ ]:
# (Opsional) pakai Google Drive
# from google.colab import drive
# drive.mount('/content/drive')
# DATA_DIR = '/content/drive/MyDrive/sibi_data'

import os, glob, json, random
import numpy as np, pandas as pd

DATA_DIR = '/content/data'
EXTRA_CSV_GLOB = os.path.join(DATA_DIR, '*.csv')   # rekaman sendiri
EXCLUDE = {'J', 'Z'}                                 # isyarat bergerak, ditangani terpisah
SEED = 42
random.seed(SEED); np.random.seed(SEED)

FEAT_COLS = [f'{a}{i}' for i in range(21) for a in 'xyz']   # x0,y0,z0,x1,...

## 1. Ekstraksi landmark dari gambar
Koordinat disimpan dalam **piksel** (x·lebar, y·tinggi, z·lebar) supaya rasio aspek gambar persegi dan webcam 16:9 tidak membuat bentuk tangan terdistorsi.

In [ ]:
import cv2, mediapipe as mp
from mediapipe.tasks import python as mp_python
from mediapipe.tasks.python import vision
from tqdm.auto import tqdm

detector = vision.HandLandmarker.create_from_options(vision.HandLandmarkerOptions(
    base_options=mp_python.BaseOptions(model_asset_path='hand_landmarker.task'),
    running_mode=vision.RunningMode.IMAGE,
    num_hands=1,
    min_hand_detection_confidence=0.3))

IMG_EXT = ('.jpg', '.jpeg', '.png', '.bmp', '.webp')
rows, failed = [], {}

for src in sorted(os.listdir(DATA_DIR)):
    src_dir = os.path.join(DATA_DIR, src)
    if not os.path.isdir(src_dir):
        continue
    for label in sorted(os.listdir(src_dir)):
        lab_dir = os.path.join(src_dir, label)
        lab = label.strip().upper()
        if not os.path.isdir(lab_dir) or lab in EXCLUDE:
            continue
        files = [f for f in os.listdir(lab_dir) if f.lower().endswith(IMG_EXT)]
        for f in tqdm(files, desc=f'{src}/{lab}', leave=False):
            bgr = cv2.imread(os.path.join(lab_dir, f))
            if bgr is None:
                failed[(src, lab)] = failed.get((src, lab), 0) + 1; continue
            rgb = np.ascontiguousarray(cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB))
            h, w = rgb.shape[:2]
            res = detector.detect(mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb))
            if not res.hand_landmarks:
                failed[(src, lab)] = failed.get((src, lab), 0) + 1; continue
            row = {'label': lab, 'source': src, 'group': src}
            for i, p in enumerate(res.hand_landmarks[0]):
                row[f'x{i}'] = p.x * w; row[f'y{i}'] = p.y * h; row[f'z{i}'] = p.z * w
            rows.append(row)

df_img = pd.DataFrame(rows, columns=['label', 'source', 'group'] + FEAT_COLS)
df_img.to_csv('/content/landmarks_from_images.csv', index=False)
print(f'{len(df_img)} gambar berhasil, {sum(failed.values())} gagal dideteksi')
if failed:
    print('Gagal terbanyak:', sorted(failed.items(), key=lambda kv: -kv[1])[:10])

## 2. Gabungkan dengan rekaman sendiri & cek sebaran data

In [ ]:
frames = [df_img]
for p in glob.glob(EXTRA_CSV_GLOB):
    d = pd.read_csv(p)
    frames.append(d)
    print('Tambah', os.path.basename(p), len(d), 'baris')

df = pd.concat(frames, ignore_index=True)
df['label'] = df['label'].astype(str).str.upper()
df = df[~df['label'].isin(EXCLUDE)].dropna(subset=FEAT_COLS).reset_index(drop=True)

print('\nJumlah per label:'); print(df.groupby('label').size().to_string())
print('\nJumlah per group (sumber/orang):'); print(df.groupby('group').size().to_string())

## 3. Normalisasi
- Titik 0 (pergelangan) dijadikan pusat koordinat.
- Skala dibagi jarak pergelangan → pangkal jari tengah (titik 9), sehingga jarak tangan ke kamera tidak berpengaruh.
- **Rotasi tidak dinormalisasi**, karena arah tangan membedakan beberapa huruf.

Fungsi ini **harus identik** dengan versi di `record_and_test.py` dan `sibi-classifier.js`.

In [ ]:
def normalize(pts):
    pts = pts - pts[0]
    scale = float(np.linalg.norm(pts[9, :2]))
    if scale < 1e-6:
        scale = 1.0
    return (pts / scale).astype(np.float32)

X = np.stack([normalize(p) for p in df[FEAT_COLS].to_numpy(np.float32).reshape(-1, 21, 3)])
labels = sorted(df['label'].unique())
lab2id = {l: i for i, l in enumerate(labels)}
y = df['label'].map(lab2id).to_numpy()
groups = df['group'].astype(str).to_numpy()
print(X.shape, len(labels), 'kelas:', labels)

## 4. Augmentasi landmark
Cermin (tangan kiri/kanan), rotasi kecil ±15°, variasi skala, noise posisi, dan variasi kedalaman.

In [ ]:
def augment(pts, rng):
    p = pts.copy()
    if rng.random() < 0.5:
        p[:, 0] *= -1                                   # tangan kiri <-> kanan
    a = np.deg2rad(rng.uniform(-15, 15))
    c, s = np.cos(a), np.sin(a)
    x, yy = p[:, 0].copy(), p[:, 1].copy()
    p[:, 0] = c * x - s * yy
    p[:, 1] = s * x + c * yy
    p[:, :2] *= rng.uniform(0.9, 1.1)
    p[:, 2] *= rng.uniform(0.7, 1.3)
    p += rng.normal(0, 0.02, p.shape).astype(np.float32)
    return p

def make_augmented(Xs, ys, k=6, seed=SEED):
    rng = np.random.default_rng(seed)
    xs = [Xs] + [np.stack([augment(p, rng) for p in Xs]) for _ in range(k)]
    return np.concatenate(xs), np.concatenate([ys] * (k + 1))

## 5. Model & fungsi pelatihan

In [ ]:
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight

N_CLS = len(labels)

def build_model():
    reg = tf.keras.regularizers.l2(1e-4)
    m = tf.keras.Sequential([
        tf.keras.layers.Input((63,)),
        tf.keras.layers.Dense(128, activation='relu', kernel_regularizer=reg),
        tf.keras.layers.Dropout(0.3),
        tf.keras.layers.Dense(64, activation='relu', kernel_regularizer=reg),
        tf.keras.layers.Dropout(0.2),
        tf.keras.layers.Dense(N_CLS, activation='softmax'),
    ])
    m.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
              loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return m

def train(X_tr, y_tr, verbose=2):
    tf.random.set_seed(SEED)
    Xa, Xv, ya, yv = train_test_split(X_tr, y_tr, test_size=0.15, stratify=y_tr, random_state=SEED)
    Xa, ya = make_augmented(Xa, ya)
    present = np.unique(ya)
    cw = dict(zip(present, compute_class_weight('balanced', classes=present, y=ya)))
    m = build_model()
    m.fit(Xa.reshape(-1, 63), ya, validation_data=(Xv.reshape(-1, 63), yv),
          epochs=200, batch_size=128, class_weight=cw, verbose=verbose,
          callbacks=[tf.keras.callbacks.EarlyStopping(monitor='val_accuracy', patience=15, restore_best_weights=True),
                     tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', patience=6, factor=0.5)])
    return m

## 6. Latih & uji
**Paling jujur:** isi `TEST_GROUPS` dengan nama orang/sumber yang **tidak ikut dilatih** (misalnya rekaman satu teman). Split acak akan memberi angka terlalu optimistis karena frame yang mirip bisa jatuh di train dan test sekaligus.

In [ ]:
TEST_GROUPS = []   # contoh: ['budi']  atau  ['sibi_kaggle']

if TEST_GROUPS:
    te = np.isin(groups, TEST_GROUPS)
    X_tr, y_tr, X_te, y_te = X[~te], y[~te], X[te], y[te]
    print('Uji pada group yang tidak dilatih:', TEST_GROUPS)
else:
    print('PERINGATAN: split acak -> akurasi cenderung terlalu optimistis')
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, stratify=y, random_state=SEED)

model = train(X_tr, y_tr)

## 7. Evaluasi: akurasi, laporan per huruf, confusion matrix

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

pred = model.predict(X_te.reshape(-1, 63), verbose=0).argmax(1)
print(f'Akurasi uji: {(pred == y_te).mean():.4f}  ({len(y_te)} sampel)\n')
print(classification_report(y_te, pred, labels=range(N_CLS), target_names=labels, digits=3, zero_division=0))

cm = confusion_matrix(y_te, pred, labels=range(N_CLS))
fig, ax = plt.subplots(figsize=(13, 13))
ConfusionMatrixDisplay(cm, display_labels=labels).plot(ax=ax, cmap='Blues', colorbar=False, xticks_rotation=90)
ax.set_title('Confusion Matrix - Abjad Jari & Angka SIBI')
plt.tight_layout(); plt.savefig('/content/confusion_matrix.png', dpi=150); plt.show()

off = cm.copy(); np.fill_diagonal(off, 0)
print('Pasangan yang paling sering tertukar (asli -> tebakan):')
for flat in np.argsort(-off.ravel())[:10]:
    i, j = np.unravel_index(flat, off.shape)
    if off[i, j]:
        print(f'  {labels[i]} -> {labels[j]}: {off[i, j]}')

## 8. (Opsional, sangat dianjurkan) Leave-one-group-out
Latih ulang sekali per group, setiap kali menguji pada group yang disisihkan. Angka rata-rata inilah yang paling layak ditulis di proposal. Butuh minimal 3 group.

In [ ]:
uniq = sorted(set(groups))
if len(uniq) >= 3:
    scores = {}
    for g in uniq:
        te = groups == g
        m = train(X[~te], y[~te], verbose=0)
        acc = (m.predict(X[te].reshape(-1, 63), verbose=0).argmax(1) == y[te]).mean()
        scores[g] = acc
        print(f'{g:25s} akurasi = {acc:.4f}  ({te.sum()} sampel)')
    print(f'\nRata-rata LOGO: {np.mean(list(scores.values())):.4f}  ± {np.std(list(scores.values())):.4f}')
else:
    print('Butuh minimal 3 group untuk LOGO. Tambahkan rekaman dari beberapa orang.')

## 9. Latih model final dengan seluruh data & ekspor
Setelah evaluasi selesai, model final dilatih memakai **semua** data, lalu bobotnya diekspor ke JSON.

In [ ]:
final_model = train(X, y, verbose=0)

export = {
    'labels': labels,
    'input_size': 63,
    'normalization': 'pixel coords (x*w, y*h, z*w); origin = landmark 0; scale = |p9 - p0| on xy',
    'layers': [],
}
for layer in final_model.layers:
    if isinstance(layer, tf.keras.layers.Dense):
        W, b = layer.get_weights()          # W: (in, out)
        export['layers'].append({
            'W': np.round(W, 6).tolist(),
            'b': np.round(b, 6).tolist(),
            'activation': layer.activation.__name__,
        })

with open('/content/sibi_mlp.json', 'w') as f:
    json.dump(export, f)
final_model.save('/content/sibi_mlp.keras')
print('Tersimpan: sibi_mlp.json,', os.path.getsize('/content/sibi_mlp.json') // 1024, 'KB')

from google.colab import files
files.download('/content/sibi_mlp.json')
files.download('/content/confusion_matrix.png')